# Interactive Algorithm Comparison Dashboard

Click legend entries to toggle algorithms. Double-click to isolate one.

In [13]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

run = ["aisle_first_best_30_reps"]

"""
Supported values for `run`:
- str: single run name (e.g., 'a_20260421_195503') or direct CSV path
- list[str]: multiple run names and/or direct CSV paths
- []: auto-detect and load all CSV files under results/
- None: auto-detect the latest CSV file under results/
"""

results_candidates = [Path("results"), Path("../results")]
results_dir = next((p for p in results_candidates if p.exists() and p.is_dir()), None)
if results_dir is None:
    searched = [str(p.resolve()) for p in results_candidates]
    raise FileNotFoundError(f"Results folder not found. Searched: {searched}")


def resolve_csv_path(entry: str) -> Path:
    entry_path = Path(entry)
    # If a direct CSV path is provided, try it as-is and relative to the results folder.
    if entry_path.suffix.lower() == ".csv" or "/" in entry or "\\" in entry:
        direct_candidates = [entry_path, results_dir / entry_path]
        for candidate in direct_candidates:
            if candidate.exists() and candidate.is_file():
                return candidate

        searched = [str(p.resolve()) for p in direct_candidates]
        raise FileNotFoundError(
            f"CSV not found for run entry {entry!r}. Searched: {searched}"
        )

    run_dir = results_dir / entry
    if run_dir.exists() and run_dir.is_dir():
        summary_candidates = sorted(run_dir.glob("summary_*.csv"))
        if summary_candidates:
            return summary_candidates[0]  # first summary_ file in the folder

    # Fallback to legacy expected filename.
    direct_candidates = [run_dir / f"summary_{entry}.csv"]
    for candidate in direct_candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    searched = [str(p.resolve()) for p in direct_candidates]
    raise FileNotFoundError(
        f"CSV not found for run entry {entry!r}. Searched: {searched}"
    )


csv_files: list[Path]

if run is None:
    csv_candidates = sorted(results_dir.rglob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError(f"No .csv files found inside {results_dir.resolve()}")
    csv_files = [csv_candidates[-1]]  # latest by sorted path
elif isinstance(run, str):
    csv_files = [resolve_csv_path(run)]
elif isinstance(run, list):
    if len(run) == 0:
        csv_files = sorted(results_dir.rglob("*.csv"))
        if not csv_files:
            raise FileNotFoundError(
                f"No .csv files found inside {results_dir.resolve()}"
            )
    else:
        csv_files = [resolve_csv_path(entry) for entry in run]
else:
    raise TypeError("`run` must be None, str, or list[str].")

print("Using CSV files:")
for file_path in csv_files:
    print(f"- {file_path}")

df = pd.concat([pd.read_csv(path) for path in csv_files], ignore_index=True)

required_columns = ["instance", "algorithm", "objective_mean", "exec_time_mean"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

best_objectives_candidates = [
    Path("best_solutions") / "best_objectives.csv",
    Path("../best_solutions") / "best_objectives.csv",
]
best_objectives_path = next((p for p in best_objectives_candidates if p.exists()), None)
if best_objectives_path is None:
    searched = [str(p.resolve()) for p in best_objectives_candidates]
    raise FileNotFoundError(f"Best objectives file not found. Searched: {searched}")

best_df = pd.read_csv(best_objectives_path)
if "instance" not in best_df.columns or "best_objective" not in best_df.columns:
    raise ValueError(
        "best_objectives.csv must contain columns: instance, best_objective"
    )

if "dataset" in df.columns and "dataset" in best_df.columns:
    merge_columns = ["dataset", "instance"]
else:
    merge_columns = ["instance"]

df = df.merge(best_df[merge_columns + ["best_objective"]], on=merge_columns, how="left")
ordered_columns = [col for col in df.columns if col != "best_objective"] + [
    "best_objective"
]
df = df[ordered_columns]


def add_legend_mass_toggle(fig: go.Figure, y_position: float = 1.18) -> None:
    trace_count = len(fig.data)
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=1.0,
                y=y_position,
                xanchor="right",
                yanchor="bottom",
                showactive=False,
                buttons=[
                    dict(
                        label="Selecionar todos",
                        method="update",
                        args=[{"visible": [True] * trace_count}],
                    ),
                    dict(
                        label="Remover todos",
                        method="update",
                        args=[{"visible": ["legendonly"] * trace_count}],
                    ),
                ],
            )
        ]
    )


df.head()

Using CSV files:
- ../results/aisle_first_best_30_reps/summary_a_20260427_201856.csv


,dataset,algorithm,instance,total_runs,feasible_runs,infeasible_runs,timed_out_runs,objective_mean,objective_median,objective_mode,...,aisles_variance,aisles_std_dev,exec_time_mean,exec_time_median,exec_time_mode,exec_time_min,exec_time_max,exec_time_variance,exec_time_std_dev,best_objective
0,a,af_useful_order-desc_prune-multi,instance_0001.txt,1,1,0,0,15.000000,15.000000,15.000000,...,0.0,0.0,0.000911,0.000911,0.000911,0.000911,0.000911,0.0,0.0,15.000
1,a,af_useful_order-desc_prune-multi,instance_0002.txt,1,1,0,0,2.000000,2.000000,2.000000,...,0.0,0.0,0.000228,0.000228,0.000228,0.000228,0.000228,0.0,0.0,2.000
2,a,af_useful_order-desc_prune-multi,instance_0003.txt,1,1,0,0,6.888889,6.888889,6.888889,...,0.0,0.0,0.003851,0.003851,0.003851,0.003851,0.003851,0.0,0.0,12.000
3,a,af_useful_order-desc_prune-multi,instance_0004.txt,1,1,0,0,3.500000,3.500000,3.500000,...,0.0,0.0,0.000700,0.000700,0.000700,0.000700,0.000700,0.0,0.0,3.500
4,a,af_useful_order-desc_prune-multi,instance_0005.txt,1,1,0,0,149.680000,149.680000,149.680000,...,0.0,0.0,0.174245,0.174245,0.174245,0.174245,0.174245,0.0,0.0,177.875


In [14]:
# Objective Mean as % of Best Objective

objective_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='objective_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

best_objective_by_instance = (
    df[['instance', 'best_objective']]
    .drop_duplicates(subset='instance')
    .set_index('instance')['best_objective']
)

objective_pct_plot = objective_plot.div(best_objective_by_instance, axis=0) * 100
objective_pct_plot = objective_pct_plot.replace([float('inf'), -float('inf')], 0).fillna(0)

fig = go.Figure()
for algo in objective_pct_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=objective_pct_plot.index,
        y=objective_pct_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.1f}}% of best<extra></extra>'
    ))

fig.add_hline(y=100, line_dash='dash', line_color='black', annotation_text='best (100%)')

fig.update_layout(
    barmode='group',
    title='Objective Mean by Instance and Algorithm (% of Best Objective)',
    xaxis_title='Instance',
    yaxis_title='% of best_objective',
    legend_title='Algorithm',
    height=700,
    margin=dict(t=150),
    xaxis_tickangle=-45,
)
add_legend_mass_toggle(fig)
fig.show()

In [15]:
# Execution Time Mean

exec_time_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='exec_time_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

fig = go.Figure()
for algo in exec_time_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=exec_time_plot.index,
        y=exec_time_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.4f}}s<extra></extra>'
    ))

fig.update_layout(
    barmode='group',
    title='Execution Time Mean by Instance and Algorithm',
    xaxis_title='Instance',
    yaxis_title='exec_time_mean (s)',
    legend_title='Algorithm',
    height=600,
    margin=dict(t=150),
    xaxis_tickangle=-45,
)
add_legend_mass_toggle(fig)
fig.show()

In [16]:
# Ranking baseado na porcetagem do melhor objetivo
# Para cada algortimo, soma a porcetagem do melhor objetivo em cada instancia e divide pela quantidade de instancias. Quanto maior, melhor

ranking_df = objective_pct_plot.mean().sort_values()
ranking_df = ranking_df.reset_index()
ranking_df.columns = ['algorithm', 'avg_pct_of_best']
ranking_df['rank'] = ranking_df['avg_pct_of_best'].rank(ascending=False, method='min')
ranking_df = ranking_df.sort_values('rank')
ranking_df


,algorithm,avg_pct_of_best,rank
2,afe_useful_order-desc_prune-multi,90.825104,1.0
3,afe_useful_order-null_prune-multi,90.825104,1.0
1,af_useful_order-desc_prune-multi,90.812736,3.0
0,af_useful_order-null_prune-multi,89.682602,4.0


In [17]:
# Tempo total por algoritmo (soma de exec_time_mean)

total_exec_time_by_algorithm = (
    df.groupby('algorithm', as_index=False)['exec_time_mean']
    .sum()
    .rename(columns={'exec_time_mean': 'total_exec_time_mean'})
    .sort_values('total_exec_time_mean', ascending=False)
    .reset_index(drop=True)
)

total_exec_time_by_algorithm

,algorithm,total_exec_time_mean
0,afe_useful_order-null_prune-multi,55.736790
1,afe_useful_order-desc_prune-multi,39.092808
2,af_useful_order-null_prune-multi,2.811793
3,af_useful_order-desc_prune-multi,2.561708
